# Notebook 04 — Model Training & Evaluation
**Sugarcane Leaf Disease Detection using Deep Feature Fusion**

This notebook covers:
- XGBoost classifier training on 1,291-dim fused feature vectors
- Accuracy, per-class precision/recall/F1 evaluation
- Confusion matrix visualisation
- XGBoost feature importance analysis

**Final Result: 92.36% overall accuracy across 12 disease classes**


In [ ]:
import sys, os
sys.path.append(os.path.join('..', 'src'))

import numpy as np
import joblib
from model import train_xgboost, evaluate_model, save_model
from utils import plot_confusion_matrix, plot_feature_importance


In [ ]:
# ── Load fused features ───────────────────────────────────────────────────
X_train = np.load('../data/X_train_fused.npy')
X_val   = np.load('../data/X_val_fused.npy')
X_test  = np.load('../data/X_test_fused.npy')
y_train = np.load('../data/y_train.npy')
y_val   = np.load('../data/y_val.npy')
y_test  = np.load('../data/y_test.npy')

label_encoder = joblib.load('../data/label_encoder.pkl')
class_names   = list(label_encoder.classes_)

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')


In [ ]:
# ── Train XGBoost ─────────────────────────────────────────────────────────
clf = train_xgboost(
    X_train, y_train,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
)
print('Training complete.')


In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────
cm = evaluate_model(clf, X_test, y_test, class_names=class_names)


In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────
plot_confusion_matrix(cm, class_names, title='XGBoost — Confusion Matrix (Test Set)')


In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────
# Label the last 11 features as handcrafted
n_deep = 1280
feature_names = ([f'MobileNetV2_{i}' for i in range(n_deep)] +
                 ['GLCM_contrast','GLCM_dissimilarity','GLCM_homogeneity',
                  'GLCM_energy','GLCM_correlation','GLCM_ASM',
                  'GLCM_mean','GLCM_std','HSV_H','HSV_S','HSV_V'])

plot_feature_importance(clf, feature_names=feature_names, top_n=20)


In [ ]:
# ── Save model ────────────────────────────────────────────────────────────
save_model(clf, '../data/xgboost_model.pkl')
